# Moore-AnimateAnyone 실행 노트북

**실행 전 필수 확인**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 셀을 **순서대로** 실행하세요
- 셀 1 실행 후 런타임이 자동 재시작됩니다 → 정상입니다. 셀 2부터 이어서 실행하세요

## 셀 1. condacolab 설치
⚠️ 실행 후 런타임 자동 재시작됩니다. 재시작 후 셀 2부터 실행하세요.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


## 셀 2. 레포 클론
런타임 재시작 후 여기서부터 실행하세요.

In [ ]:
%cd /content
!git clone https://github.com/MooreThreads/Moore-AnimateAnyone.git
%cd Moore-AnimateAnyone

## 셀 3. Python 3.10 환경 생성 및 패키지 설치
10~15분 소요됩니다. 기다려주세요.

In [2]:
# Python 3.10 환경 생성
!conda create -n animate python=3.10 -y

# FFmpeg 개발 헤더 설치
!apt-get install -y libavformat-dev libavcodec-dev libavdevice-dev \
    libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config -q

# conda 환경 pip 버전 낮추기 (torchsde 설치 오류 방지)
!conda run -n animate pip install "pip<24.1" -q

# av 먼저 설치
!conda run -n animate pip install av==11.0.0 -q

# clip@ 줄 제거 (특정 커밋 해시로 지정돼 있어서 충돌 발생)
with open('requirements.txt', 'r') as f:
    lines = f.readlines()
with open('requirements.txt', 'w') as f:
    for line in lines:
        if not line.startswith('clip @') and not line.startswith('clip@'):
            f.write(line)

# clip 별도 설치
!conda run -n animate pip install git+https://github.com/openai/CLIP.git -q

# 나머지 전체 설치
!conda run -n animate pip install -r requirements.txt --ignore-installed clip -q

# huggingface_hub 버전 고정 (cached_download 오류 방지)
!conda run -n animate pip install huggingface_hub==0.21.0 -q

print("\n✅ 설치 완료!")

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/animate

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    icu-78.3                   |       h33c6efd_0        12.1 MB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.7.4             |       hecca717_0          75 KB  conda-forge
 

## 셀 4. 설치 확인

In [3]:
!conda run -n animate pip show torch torchvision av onnxruntime-gpu
# 확인 항목:
# torch: 2.0.1
# torchvision: 0.15.2
# av: 11.0.0
# onnxruntime-gpu: 1.16.3

Name: torch
Version: 2.0.1
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/envs/animate/lib/python3.10/site-packages
Requires: filelock, jinja2, networkx, nvidia-cublas-cu11, nvidia-cuda-cupti-cu11, nvidia-cuda-nvrtc-cu11, nvidia-cuda-runtime-cu11, nvidia-cudnn-cu11, nvidia-cufft-cu11, nvidia-curand-cu11, nvidia-cusolver-cu11, nvidia-cusparse-cu11, nvidia-nccl-cu11, nvidia-nvtx-cu11, sympy, triton, typing-extensions
Required-by: accelerate, controlnet_aux, open-clip-torch, timm, torchdiffeq, torchmetrics, torchsde, torchvision, triton, xformers
---
Name: torchvision
Version: 0.15.2
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: soumith@pytorch.org
License: BSD
Location: /usr/local/envs/animate/lib/python3.10/site-packages
R

## 셀 5. 사전학습 가중치 다운로드
수 GB 용량입니다. 시간이 걸립니다.

In [4]:
!conda run -n animate python tools/download_weights.py

Preparing base stable-diffusion-v1-5 weights...
Preparing image encoder weights...
Preparing DWPose weights...
Preparing vae weights...
Preparing AnimateAnyone weights...



## 셀 6. 예시 파일 확인

In [5]:
!find . -name "*.jpg" -o -name "*.png" -o -name "*.mp4" | head -20
# 레퍼런스 이미지: ./configs/inference/ref_images/anyone-1.png
# 포즈 영상: ./configs/inference/pose_videos/anyone-video-1_kps.mp4
# _kps 붙은 영상 = 이미 포즈 추출된 것

./assets/mini_program_maliang.png
./configs/inference/pose_images/pose-1.png
./configs/inference/talkinghead_images/1.png
./configs/inference/talkinghead_images/4.png
./configs/inference/talkinghead_images/5.png
./configs/inference/talkinghead_images/3.png
./configs/inference/talkinghead_images/2.png
./configs/inference/pose_videos/anyone-video-2_kps.mp4
./configs/inference/pose_videos/anyone-video-5_kps.mp4
./configs/inference/pose_videos/anyone-video-4_kps.mp4
./configs/inference/pose_videos/anyone-video-1_kps.mp4
./configs/inference/talkinghead_videos/3.mp4
./configs/inference/talkinghead_videos/1.mp4
./configs/inference/talkinghead_videos/4.mp4
./configs/inference/talkinghead_videos/2.mp4
./configs/inference/ref_images/anyone-11.png
./configs/inference/ref_images/anyone-1.png
./configs/inference/ref_images/anyone-10.png
./configs/inference/ref_images/anyone-3.png
./configs/inference/ref_images/anyone-5.png


## 셀 7. Config 파일 생성

In [6]:
import yaml, os

config = {
    "pretrained_base_model_path": "./pretrained_weights/stable-diffusion-v1-5",
    "pretrained_vae_path": "./pretrained_weights/sd-vae-ft-mse",
    "image_encoder_path": "./pretrained_weights/image_encoder",
    "denoising_unet_path": "./pretrained_weights/denoising_unet.pth",
    "reference_unet_path": "./pretrained_weights/reference_unet.pth",
    "pose_guider_path": "./pretrained_weights/pose_guider.pth",
    "motion_module_path": "./pretrained_weights/motion_module.pth",
    "inference_config": "./configs/inference/inference_v2.yaml",
    "weight_dtype": "fp16",
    "test_cases": {
        "./configs/inference/ref_images/anyone-1.png": [
            "./configs/inference/pose_videos/anyone-video-1_kps.mp4"
        ]
    }
}

os.makedirs("configs/prompts", exist_ok=True)
with open("configs/prompts/my_animation.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ config 생성 완료!")
print(yaml.dump(config, default_flow_style=False))

✅ config 생성 완료!
denoising_unet_path: ./pretrained_weights/denoising_unet.pth
image_encoder_path: ./pretrained_weights/image_encoder
inference_config: ./configs/inference/inference_v2.yaml
motion_module_path: ./pretrained_weights/motion_module.pth
pose_guider_path: ./pretrained_weights/pose_guider.pth
pretrained_base_model_path: ./pretrained_weights/stable-diffusion-v1-5
pretrained_vae_path: ./pretrained_weights/sd-vae-ft-mse
reference_unet_path: ./pretrained_weights/reference_unet.pth
test_cases:
  ./configs/inference/ref_images/anyone-1.png:
  - ./configs/inference/pose_videos/anyone-video-1_kps.mp4
weight_dtype: fp16



## 셀 8. 추론 실행
T4 GPU 기준 약 2~3분 소요

| 설정 | 최소 VRAM |
|------|----------|
| 384x512, 16프레임 | ~8GB |
| 512x784, 32프레임 | ~12GB |

In [21]:
import subprocess, sys

process = subprocess.Popen(
    ['/usr/local/envs/animate/bin/python', '-m', 'scripts.pose2vid',
     '--config', './configs/prompts/my_animation.yaml',
     '-W', '384', '-H', '512', '-L', '16'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/Moore-AnimateAnyone'
)

for line in process.stdout:
    print(line, end='')

process.wait()
print("return code:", process.returncode)

/usr/local/envs/animate/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
Some weights of the model checkpoint were not used when initializing UNet2DConditionModel: 
 ['conv_norm_out.weight, conv_norm_out.bias, conv_out.weight, conv_out.bias']
/usr/local/envs/animate/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/content/Moore-AnimateAnyone/src/pipelines/pipeline_

## 셀 9. 결과 확인 및 다운로드

In [22]:
from IPython.display import Video, display
from google.colab import files
import glob

output_files = glob.glob("output/**/*.mp4", recursive=True)
print("생성된 파일:", output_files)

if output_files:
    display(Video(output_files[-1], embed=True))
    files.download(output_files[-1])
else:
    print("output 폴더 확인:")
    !find . -name "*.mp4" | grep -v pose | grep -v talkinghead

생성된 파일: ['output/20260324/1254--seed_42-384x512/anyone-1_anyone-video-1_512x384_3_1254.mp4']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>